In [ ]:
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'

In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from pathlib import Path

# TensorFlow / Keras 3 / tf_keras legacy
import tensorflow as tf
import keras
from keras import layers

import tf_keras

# HuggingFace Transformers (para DeiT)
from transformers import TFViTModel

# Sklearn
from sklearn.metrics import classification_report

# Project imports
from source.functions import *

# Version check
print(f"TensorFlow: {tf.__version__}")
print(f"Keras: {keras.__version__}")

In [ ]:
IMG_SIZE = (512, 512)      # Tamanho original, redimensionado para 224 no pipeline DeiT
SEED = 42
NUM_CLASSES = 23
BATCH_SIZE = 32
AUTOTUNE = tf.data.AUTOTUNE

train_dir_path = Path("wikiart_split/train")
val_dir_path = Path("wikiart_split/val")
test_dir_path = Path("wikiart_split/test")

In [ ]:
train_ds_raw = keras.utils.image_dataset_from_directory(
    train_dir_path,
    label_mode='int',
    batch_size=None,
    image_size=IMG_SIZE,
    interpolation='bilinear',
    shuffle=True,
    verbose=False,
    seed=SEED
)

val_ds_raw = keras.utils.image_dataset_from_directory(
    val_dir_path,
    label_mode='int',
    batch_size=None,
    image_size=IMG_SIZE,
    interpolation='bilinear',
    shuffle=False,
    verbose=False,
    seed=SEED
)

test_ds_raw = keras.utils.image_dataset_from_directory(
    test_dir_path,
    label_mode='int',
    batch_size=None,
    image_size=IMG_SIZE,
    interpolation='bilinear',
    shuffle=False,
    verbose=False,
    seed=SEED
)

class_names = train_ds_raw.class_names
print(f"Number of classes: {len(class_names)}")
print(f"Classes: {class_names}")

In [ ]:
AUGMENTATION_CONFIGS = {
    'grayscale_mix_final': keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.03),
        layers.RandomGrayscale(factor=0.2),
        layers.RandomBrightness(0.15),
        layers.RandomContrast(0.15),
    ], name='aug_grayscale_mix'),
}

In [ ]:
# Load DeiT-Small weights via TFViTModel (architectural compatibility)
deit_backbone_pb = TFViTModel.from_pretrained("facebook/deit-small-patch16-224")

# Verify weights loaded correctly
_w = deit_backbone_pb.vit.embeddings.patch_embeddings.projection.weights[0].numpy()
print(f"Weight std: {_w.std():.6f} (expected ~0.03 for pretrained)")

In [ ]:
IMAGENET_MEAN = tf.constant([0.485, 0.456, 0.406], dtype=tf.float32)
IMAGENET_STD = tf.constant([0.229, 0.224, 0.225], dtype=tf.float32)

def deit_preprocess_channels_last(image):
    """DeiT/ViT normalization. Transpose feito separadamente no pipeline."""
    image = tf.cast(image, tf.float32) / 255.0
    image = (image - IMAGENET_MEAN) / IMAGENET_STD
    return image

In [ ]:
grayscale_aug = AUGMENTATION_CONFIGS['grayscale_mix_final']

def train_map_deit(image, label):
    image = tf.cast(image, tf.float32)
    image = grayscale_aug(image, training=True)
    image = deit_preprocess_channels_last(image)
    image = tf.image.resize(image, [224, 224])
    image = tf.transpose(image, (2, 0, 1))
    return image, label

def val_map_deit(image, label):
    image = tf.cast(image, tf.float32)
    image = deit_preprocess_channels_last(image)
    image = tf.image.resize(image, [224, 224])
    image = tf.transpose(image, (2, 0, 1))
    return image, label

def test_map_deit(image, label):
    image = tf.cast(image, tf.float32)
    image = deit_preprocess_channels_last(image)
    image = tf.image.resize(image, [224, 224])
    image = tf.transpose(image, (2, 0, 1))
    return image, label

train_ds_deit = (train_ds_raw
    .shuffle(10000, seed=SEED, reshuffle_each_iteration=True)
    .map(train_map_deit, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE))

val_ds_deit = (val_ds_raw
    .map(val_map_deit, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE))

test_ds_deit = (test_ds_raw
    .map(test_map_deit, num_parallel_calls=AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(AUTOTUNE))

# Sanity check
for images, labels in train_ds_deit.take(1):
    print(f"Batch images shape: {images.shape}")
    print(f"Batch labels shape: {labels.shape}")
    print(f"Images min: {images.numpy().min():.4f}, max: {images.numpy().max():.4f}")

In [ ]:
class SparseF1ScoreDeiT(keras.metrics.F1Score):
    """Versão local do SparseF1Score para DeiT. 
    Tolera labels com shape (B, 1) — comportamento do tf_keras Model.fit.
    """
    def update_state(self, y_true, y_pred, sample_weight=None):
        y_true = tf.reshape(tf.cast(y_true, tf.int32), [-1])
        y_true_onehot = tf.one_hot(y_true, depth=NUM_CLASSES)
        return super().update_state(y_true_onehot, y_pred, sample_weight)

In [ ]:
def build_deit_model_pb(num_classes=NUM_CLASSES, head_hidden=256, dropout=0.3):
    """DeiT-Small com receita paper-based: backbone trainable desde epoch 1."""
    inputs = tf_keras.Input(shape=(3, 224, 224), dtype=tf.float32, name="pixel_values")
    outputs = deit_backbone_pb.vit(inputs, training=False)
    cls_token = outputs.last_hidden_state[:, 0, :]
    x = tf_keras.layers.BatchNormalization()(cls_token)
    x = tf_keras.layers.Dense(head_hidden, activation='swish')(x)
    x = tf_keras.layers.Dropout(dropout)(x)
    outputs = tf_keras.layers.Dense(num_classes, activation='softmax')(x)
    return tf_keras.Model(inputs=inputs, outputs=outputs, name='DeiT_Small_paper_based')

deit_model_pb = build_deit_model_pb()

# Single-phase: backbone trainable desde o início (paper-based recipe)
deit_backbone_pb.trainable = True

trainable = sum(tf_keras.backend.count_params(w) for w in deit_model_pb.trainable_weights)
print(f"Trainable params (all end-to-end): {trainable:,}")

In [ ]:
# Paper-based hyperparameters
EPOCHS_PB = 20
STEPS_PER_EPOCH = 313
TOTAL_STEPS = EPOCHS_PB * STEPS_PER_EPOCH

# Cosine decay: 5e-5 → ~5e-7
lr_schedule_pb = tf_keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=5e-5,
    decay_steps=TOTAL_STEPS,
    alpha=0.01,
)

deit_model_pb.compile(
    optimizer=tf_keras.optimizers.AdamW(learning_rate=lr_schedule_pb, weight_decay=0.01),
    loss=tf_keras.losses.SparseCategoricalCrossentropy(),
    metrics=[
        tf_keras.metrics.SparseCategoricalAccuracy(name='accuracy'),
        tf_keras.metrics.SparseTopKCategoricalAccuracy(k=3, name='top3_accuracy'),
        SparseF1ScoreDeiT(average='macro', name='macro_f1'),
    ]
)

print(f"Compiled with AdamW (weight_decay=0.01) + CosineDecay (5e-5 → 5e-7)")
print(f"Total training steps: {TOTAL_STEPS}")

In [ ]:
callbacks_pb = [
    tf_keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1,
    ),
]

print("=" * 60)
print("DeiT-Small paper-based: single-phase end-to-end")
print(f"AdamW weight_decay=0.01, cosine decay 5e-5 → 5e-7, {EPOCHS_PB} epochs")
print("=" * 60)

history_pb = deit_model_pb.fit(
    train_ds_deit,
    validation_data=val_ds_deit,
    epochs=EPOCHS_PB,
    callbacks=callbacks_pb,
    verbose=1,
)

In [ ]:
# Carregar weights salvos em disco (em vez de re-treinar)
deit_model_pb.load_weights('checkpoints/DeiT_Small_paper_based.weights.h5')
print("Weights loaded from: checkpoints/DeiT_Small_paper_based.weights.h5")

# Carregar history também
with open('checkpoints/DeiT_Small_paper_based_history.pkl', 'rb') as f:
    history_data = pickle.load(f)

# Compatibilizar formato para plots (objeto com .history)
class LoadedHistory:
    def __init__(self, history_dict):
        self.history = history_dict

history_pb = LoadedHistory(history_data['history'])
print(f"History loaded. Epochs: {len(history_pb.history['loss'])}")
print(f"Config: {history_data['config']}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

epochs_range = range(1, len(history_pb.history['loss']) + 1)

# Loss
axes[0].plot(epochs_range, history_pb.history['loss'], label='Train', marker='o', markersize=4)
axes[0].plot(epochs_range, history_pb.history['val_loss'], label='Validation', marker='s', markersize=4)
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Macro F1
axes[1].plot(epochs_range, history_pb.history['macro_f1'], label='Train', marker='o', markersize=4)
axes[1].plot(epochs_range, history_pb.history['val_macro_f1'], label='Validation', marker='s', markersize=4)
axes[1].set_title('Macro F1')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro F1')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# Top-3 Accuracy
axes[2].plot(epochs_range, history_pb.history['top3_accuracy'], label='Train', marker='o', markersize=4)
axes[2].plot(epochs_range, history_pb.history['val_top3_accuracy'], label='Validation', marker='s', markersize=4)
axes[2].set_title('Top-3 Accuracy')
axes[2].set_xlabel('Epoch')
axes[2].set_ylabel('Top-3 Accuracy')
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.suptitle('DeiT-Small Paper-Based: Learning Curves', fontsize=14)
plt.tight_layout()
plt.savefig('figures/deit_paper_based_learning_curves.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBest val_macro_f1: {max(history_pb.history['val_macro_f1']):.4f}")
print(f"Best val_loss: {min(history_pb.history['val_loss']):.4f}")

In [ ]:
print("=" * 60)
print("DeiT-Small paper-based evaluation on TEST set")
print("=" * 60)

test_results = deit_model_pb.evaluate(test_ds_deit, verbose=1)

print("\nTest set results:")
for name, value in zip(deit_model_pb.metrics_names, test_results):
    print(f"  {name}: {value:.4f}")

In [ ]:
y_true_test = []
y_pred_test = []

print("Running predictions on test set...")
for images, labels in test_ds_deit:
    preds = deit_model_pb.predict(images, verbose=0)
    y_true_test.extend(labels.numpy())
    y_pred_test.extend(np.argmax(preds, axis=1))

y_true_test = np.array(y_true_test)
y_pred_test = np.array(y_pred_test)

print("\nPer-class classification report (TEST set):")
print(classification_report(y_true_test, y_pred_test, target_names=class_names, digits=4))

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns

cm = confusion_matrix(y_true_test, y_pred_test)

plt.figure(figsize=(14, 12))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names,
            cbar_kws={'label': 'Count'})
plt.title('DeiT-Small Paper-Based: Confusion Matrix (Test Set)', fontsize=14)
plt.xlabel('Predicted')
plt.ylabel('True')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.savefig('figures/deit_paper_based_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()